# LpWM: dense state, sparse generator

This notebook runs the slot-free PushT experiment from `feature/sparse-generator`. The representation is a dense signed 8x8 patch field; exact top-k sparsity is used only for shared dynamics laws and token relations. Start with the smoke test before spending compute units.

In [ ]:
import os

if not os.path.exists('/content/lpworldmodel'):
    !git clone --branch feature/sparse-generator https://github.com/twojtys137/lpworldmodel.git /content/lpworldmodel
%cd /content/lpworldmodel
!git fetch origin feature/sparse-generator
!git checkout feature/sparse-generator
!git pull --ff-only origin feature/sparse-generator

In [ ]:
!pip -q install 'accelerate>=0.26,<2' 'hydra-core>=1.3,<2' 'omegaconf>=2.3,<3' 'wandb>=0.13,<1' einops decord pymunk pygame shapely scikit-image moviepy tensorboardX
!python -m pytest -q tests/test_sparse_generator.py

## Dataset

Download the DINO-WM PushT data linked in the repository README and place it in Drive so that `DATASET_DIR/pusht_noise/train` and `DATASET_DIR/pusht_noise/val` exist.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATASET_DIR = '/content/drive/MyDrive/lpwm-data'  # change if needed
assert os.path.isdir(os.path.join(DATASET_DIR, 'pusht_noise', 'train')), DATASET_DIR
os.environ['DATASET_DIR'] = DATASET_DIR
os.environ['WANDB_MODE'] = 'offline'

## 1. Smoke run

Eight rollouts, one epoch, batch 4 and 64 RDMReg projections. This validates data loading, gradients, checkpointing, and GPU memory.

In [ ]:
!SMOKE=1 RUN_NAME=sparse_generator bash scripts/train_sparse_generator_colab.sh

## 2. Screening run

The default is 50 rollouts, two epochs, batch 16, 64 patches, D=192, M=8, law top-k=2 and edge top-k=8. Set the flag only after the smoke run succeeds.

In [ ]:
RUN_FULL = False
if RUN_FULL:
    !RUN_NAME=sparse_generator_seed0 SEED=0 bash scripts/train_sparse_generator_colab.sh

## 3. Controlled ablations

Keep the encoder and dense identity-linked state fixed. Compare `ltv`, `sparse_ltv`, `dense_generator`, and `sparse_generator`; promote only the best two to seeds 1 and 2. The detailed experiment matrix and interpretation of routing diagnostics are in `docs/sparse_generator.md`.

In [ ]:
# Print the matched commands; add RUN=1 (and initially SMOKE=1) to execute:
!bash scripts/sweep_sparse_generator_colab.sh
# !RUN=1 SMOKE=1 bash scripts/sweep_sparse_generator_colab.sh

# Example minimal sparse-LTV control using the same matched Colab config:
# !PREDICTOR=sparse_ltv NUM_PROJECTIONS=256 N_ROLLOUT=50 \
#   RUN_NAME=sparse_ltv_seed0 bash scripts/train_sparse_generator_colab.sh